In [7]:
"""
Supply Chain Risk Analyst — Tightly Coupled Version
====================================================
This version works. It also locks you in.

Read the comments marked with [LOCK-IN] to see why.
"""

import os
import json
from openai import OpenAI                 # [LOCK-IN #1A] Direct import
from openai.types.chat import (
    ChatCompletionMessageParam,
    ChatCompletionToolParam,
    ChatCompletionAssistantMessageParam,
)                                         # [LOCK-IN #1B] Direct import
from typing import cast
from tavily import TavilyClient
from datetime import datetime
# Text-wrap for cell outputs
import textwrap
def wrap(text, width=180):
    """Wrap text for clean display in notebook output."""
    return "\n".join(textwrap.fill(line, width=width) if line.strip() else line
                     for line in text.splitlines())

# [LOCK-IN #2] Client instantiated at module level, hardcoded provider
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# [LOCK-IN #3] Model name hardcoded as a constant
MODEL = "gpt-4o-mini"

# Using a similar prompt so we do not purposefully sabotage this version of the agent
today = datetime.now().strftime("%B %Y")

SYSTEM_PROMPT = f"""You are a Supply Chain Risk Analyst. Today's date is {today}.

CRITICAL RULES:
1. You MUST call the search tool at least once before giving a final answer.
2. Never answer from memory — always search first.
3. Always include the current month and year in your search queries.
4. If a search returns irrelevant results, try ONE different query then move on.
5. Never repeat the same search query twice.
6. After 3 tool calls, you MUST write your Final Answer using whatever you found

Format your final answer as:
SEVERITY: [LOW|MEDIUM|HIGH]
SUMMARY: [2-3 sentences based on search results]
RECOMMENDATION: [one specific action]
"""


def search_supply_chain(query: str) -> str:
    """Wrapper around Tavily search. Provider-agnostic — this part is fine."""
    try:
        result = tavily.search(query=query, search_depth="advanced", max_results=5)
        return json.dumps(result.get("results", []))
    except Exception as e:
        return f"Search failed: {e}"


# [LOCK-IN #4] Tool schema written in OpenAI's specific function-calling format.
# Anthropic uses a different shape. Google uses a third. This dict is OpenAI-only.
TOOLS: list[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "search_supply_chain",
            "description": "Search for current supply chain news and disruptions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }
    }
]


def run_agent(user_question: str, max_turns: int = 4) -> str:
    """
    Run the agent loop. Returns the final answer string.

    [LOCK-IN #5] The entire loop assumes OpenAI's message and tool_calls structure.
    Switching providers means rewriting this function from scratch.
    """
    messages: list[ChatCompletionMessageParam] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_question}
    ]

    for turn in range(max_turns):
        try:
            # [LOCK-IN #6] OpenAI-specific call signature
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=TOOLS,
                tool_choice="auto"
            )
        except Exception as e:
            return f"LLM call failed on turn {turn + 1}: {e}"

        msg = response.choices[0].message
        # Convert the response message into an assistant input message before appending.
        # The shapes are similar but technically distinct types in the SDK.
        assistant_msg: ChatCompletionAssistantMessageParam = {
            "role": "assistant",
            "content": msg.content,
        }
        if msg.tool_calls:
            # Re-serialize tool_calls into the input format
            assistant_msg["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in msg.tool_calls
                if tc.type == "function"  # narrow the discriminated union
            ]
        messages.append(assistant_msg)

        # If no tool calls, we're done
        if not msg.tool_calls:
            return msg.content or "No answer produced."

        # [LOCK-IN #7] Tool dispatch — narrow the union before accessing .function
        for tool_call in msg.tool_calls:
            if tool_call.type != "function":
                # Skip custom tool calls — we only handle function tools in this example
                continue

            try:
                args = json.loads(tool_call.function.arguments)
                if tool_call.function.name == "search_supply_chain":
                    result = search_supply_chain(args["query"])
                else:
                    result = f"Unknown tool: {tool_call.function.name}"
            except Exception as e:
                result = f"Tool execution failed: {e}"

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

    return "Max turns reached without final answer."


# Example usage
if __name__ == "__main__":
    answer = run_agent('How is canada\'s supply chain impacted by the Strait of Hormuz blockades in April 2026?')
    print(wrap(answer))

SEVERITY: HIGH
SUMMARY: The ongoing blockade of the Strait of Hormuz has led to significant disruptions in global supply chains, particularly affecting oil prices, fertilizers, and food supplies,
which are integral to Canada's economy. As approximately 30% of the world’s fertilizer traverses this strait, Canadian farmers and consumers can expect dramatic increases in food
prices and inflation, severely impacting food security and agricultural productivity in the country. The blockade has also resulted in heightened energy prices, compelling the
Canadian government to suspend fuel taxes to mitigate economic strain on consumers.
RECOMMENDATION: Canadian stakeholders should urgently assess alternative supply routes for essential goods and consider strategic reserves to manage expected shortages and price
volatility in the agriculture and energy sectors.
